<h1><center>Building a Neural Network from Scratch</center></h1>
<hr>

<h1>Objective</h1>
<p>Building a 3 layer neural network (input -> hidden -> output) using only numpy and math to classify images - each image having 28x28 pixels, and each pixel having a grayscale value between 0 and 255 -  of handwritten digits from the MNIST dataset.</p>
<hr>

<h2>The Math:</h2>

**Forward Propogation:**
Computes the 'Z' matrix of the [1] or hidden layer from the 'X' matrix of the [0] or input layer by multiplying the transposed input matrix 'X' (so each column has 784 rows) with weights and adding a bias. This 'Z' matrix is then passed through a ReLU function. To compute the [2] or output layer from the [1] or hidden layer we utilise the Softmax activation function.

**Backward Propogation:**
Computes the gradients for updating weights and biases, working backwards from the [2] or output layer. dZ[2] is found by subtracting the true labels 'Y' from the predicted output 'A[2]'. This gives dW[2] and db[2] by multiplying dZ[2] with the transposed hidden layer activations and averaging over the m training examples. To propagate the error to the [1] or hidden layer, dZ[1] is computed by multiplying the transposed weights W[2] with dZ[2], then taking the element-wise product with the derivative of ReLu applied to Z[1]. Finally, dW[1] and db[1] are computed the same way, using dZ[1] and the transposed input matrix 'X', averaged over m.

Learning Rate: Parameters are then adequately updated using what was learned during both propogations.

W[1] = W[1] - @dW[1] (@ = alpha)

b[1] = b[1] - @db[1]

and so on in a continuous process.

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [26]:
df = pd.read_csv('train.csv')

In [27]:
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [28]:
df = np.array(df)
m, n = df.shape
np.random.shuffle(df)

df_dev = df[0:1000].T
Y_dev = df_dev[0]
X_dev = df_dev[1:n]

df_train = df[1000:m].T
Y_train = df_train[0]
X_train = df_train[1:n]

**Initialize starting Parameters**

In [29]:
def init_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

**Forward Propogation**

In [30]:
def ReLU(Z):
    return np.maximum(0,Z)

def softmax(Z):
    Z = Z - np.max(Z, axis=0, keepdims=True)  # stability
    return np.exp(Z) / np.sum(np.exp(Z), axis=0, keepdims=True)

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

**Backward Propogation**

In [31]:
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def deriv_ReLU(Z):
    return Z > 0

def back_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

**Update parameters (Learning Rate):**

In [32]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

**Gradient Descent:**

In [33]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def GD(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if (i % 10 == 0):
            print("Iteration: ", i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))
    return W1, b1, W2, b2

In [34]:
W1, b1, W2, b2  = GD(X_train, Y_train, 100, 0.1) #100 iterations, learning rate = 0.1

Iteration:  0
[7 7 7 ... 7 7 8] [2 2 1 ... 2 0 4]
Accuracy:  0.09665853658536586
Iteration:  10
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10021951219512196
Iteration:  20
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10021951219512196
Iteration:  30
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10021951219512196
Iteration:  40
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10021951219512196
Iteration:  50
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10021951219512196
Iteration:  60
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10021951219512196
Iteration:  70
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10021951219512196
Iteration:  80
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10026829268292684
Iteration:  90
[9 9 9 ... 9 9 9] [2 2 1 ... 2 0 4]
Accuracy:  0.10026829268292684
